# Seafloor W-face: 0.0 vs NaN - what happens to the particles?

`Fix_W` fills the masked seafloor W-face with `0.0` (`fillna(0.0)`). Particles
subduct onto that face, read W=0, and **freeze** there (no bottom kernel to lift
them out). The question: if we instead leave that face **NaN** (as the raw model
does), do the particles behave better - or does Parcels simply **delete** them?

Controlled A/B test: same 100 particles seeded just above the seafloor on the
Guiana shelf, run twice with identical everything **except** the value at the
seafloor W-face - `0.0` in one fieldset, `NaN` in the other. We count how many
**survive** (frozen at the bottom) vs are **deleted** (Parcels errors on the NaN
and `CheckError` removes them).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import xarray as xr
from datetime import timedelta
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from parcels import (FieldSet, ParticleSet, ScipyParticle, AdvectionRK4_3D,
                     StatusCode)
import config as C

# --- TUNE ME -----------------------------------------------------------------
MONTHS   = ["2005-06", "2005-07"]
SEED_BOX = dict(lon=(-53.0, -49.5), lat=(5.0, 8.0), max_levels=15)
N_PART   = 100
RUN_DAYS = 8
DT_MIN   = 20
SEED_ABOVE_BOTTOM = 1.0        # release this far above the seafloor (m)
# -----------------------------------------------------------------------------

UVW  = "/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW"
Hgr  = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
MESHZ = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
TMP  = os.path.abspath("./_wnan_tmp"); os.makedirs(TMP, exist_ok=True)
print("months:", MONTHS, "| N:", N_PART, "| run:", RUN_DAYS, "d")

## Make a NaN-seafloor copy of the W files
For every wet column, set the seafloor face (depth index = `mbathy`) back to
`NaN` - i.e. undo exactly the `fillna(0.0)` that `Fix_W` applied there. Nothing
else is touched.

In [ ]:
Zg = xr.open_dataset(MESHZ)
mbathy = np.asarray(Zg["mbathy"]).squeeze().astype(int)
gdept  = np.asarray(Zg["gdept_0"]).squeeze()
gdepw  = np.asarray(Zg["gdepw_0"]).squeeze()
yj, xi = np.where(mbathy > 0)
kk = mbathy[yj, xi]                       # seafloor face index per wet column

wnan_files = []
for m in MONTHS:
    src = f"{UVW}/W_{m}fc.nc"
    ds = xr.open_dataset(src)
    w = ds["vovecrtz"].values.copy()      # (time, depth, y, x)
    before = int(np.sum(~np.isnan(w)))
    w[:, kk, yj, xi] = np.nan             # NaN the seafloor face
    after = int(np.sum(~np.isnan(w)))
    ds["vovecrtz"] = (ds["vovecrtz"].dims, w)
    dst = f"{TMP}/W_{m}fc_NANfloor.nc"
    ds.to_netcdf(dst); wnan_files.append(dst)
    print(f"{m}: removed {before-after:,} seafloor-face values (set to NaN)")
print("NaN-seafloor W written to", TMP)

## Build the two fieldsets
Identical U, V; only W differs (0.0-floor vs NaN-floor).

In [ ]:
def build(wfiles):
    fn = {"U": {"data": [f"{UVW}/U_{m}c.nc" for m in MONTHS], "lon": Hgr, "lat": Hgr,
                "depth": f"{UVW}/W_{MONTHS[0]}fc.nc"},
          "V": {"data": [f"{UVW}/V_{m}c.nc" for m in MONTHS], "lon": Hgr, "lat": Hgr,
                "depth": f"{UVW}/W_{MONTHS[0]}fc.nc"},
          "W": {"data": wfiles, "lon": Hgr, "lat": Hgr,
                "depth": f"{UVW}/W_{MONTHS[0]}fc.nc"}}
    return FieldSet.from_netcdf(
        fn, {"U": "vozocrtx", "V": "vomecrty", "W": "vovecrtz"},
        {k: {"lon": "glamf", "lat": "gphif", "depth": "depthw", "time": "time_counter"}
         for k in ("U", "V", "W")},
        interp_method={k: "cgrid_velocity" for k in ("U", "V", "W")}, mesh="spherical")

fs_zero = build([f"{UVW}/W_{m}fc.nc" for m in MONTHS])   # seafloor face = 0.0 (Fix_W)
fs_nan  = build(wnan_files)                               # seafloor face = NaN
print("both fieldsets built")

## Seed 100 particles just above the seafloor

In [ ]:
Hg = xr.open_dataset(Hgr)
glamt = np.asarray(Hg["glamt"]).squeeze(); gphit = np.asarray(Hg["gphit"]).squeeze()
shelf = ((glamt > SEED_BOX["lon"][0]) & (glamt < SEED_BOX["lon"][1])
         & (gphit > SEED_BOX["lat"][0]) & (gphit < SEED_BOX["lat"][1])
         & (mbathy > 0) & (mbathy < SEED_BOX["max_levels"]))
jj, ii = np.where(shelf)
rng = np.random.default_rng(C.RANDOM_STATE)
s = rng.choice(len(jj), N_PART, replace=len(jj) < N_PART)
lon0 = glamt[jj[s], ii[s]]; lat0 = gphit[jj[s], ii[s]]
floor0 = gdepw[mbathy[jj[s], ii[s]]]
dep0 = np.clip(floor0 - SEED_ABOVE_BOTTOM, 1.0, None)
print(f"seeded {N_PART} | seafloor {floor0.min():.1f}..{floor0.max():.1f} m | "
      f"release depth {dep0.min():.1f}..{dep0.max():.1f} m")

## Run both - same kernels, only W differs
`CheckError` deletes any particle that errors (state >= 50), which is how a NaN
sample is removed - exactly as in the production run.

In [ ]:
def CheckError(particle, fieldset, time):
    if particle.state >= 50:
        particle.delete()

def run(fs, tag):
    t0 = fs.U.grid.time[0]
    ps = ParticleSet(fs, pclass=ScipyParticle, lon=lon0.copy(), lat=lat0.copy(),
                     depth=dep0.copy(), time=[t0] * N_PART)
    ps.execute([AdvectionRK4_3D, CheckError],
               runtime=timedelta(days=RUN_DAYS), dt=timedelta(minutes=DT_MIN))
    survived = len(ps)
    depths = np.array([p.depth for p in ps]) if survived else np.array([])
    print(f"  {tag:12s}: {survived:3d}/{N_PART} survived, {N_PART-survived:3d} deleted")
    return survived, depths

print("seafloor face = 0.0  (current Fix_W):")
surv0, dep_z = run(fs_zero, "0.0 floor")
print("seafloor face = NaN  (proposed):")
survn, dep_n = run(fs_nan, "NaN floor")

## Result

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["0.0 floor\n(Fix_W)", "NaN floor\n(proposed)"],
          [surv0, survn], color=["#2a78d6", "#eb6834"])
ax[0].set_ylabel("particles surviving"); ax[0].set_ylim(0, N_PART)
for k, v in enumerate([surv0, survn]):
    ax[0].text(k, v + 2, f"{v}/{N_PART}", ha="center", fontsize=10)
ax[0].set_title("survive vs deleted", loc="left", fontsize=10)
if len(dep_z):
    ax[1].hist(dep_z, bins=20, color="#2a78d6", alpha=.8, label="0.0 floor survivors")
ax[1].set_xlabel("final depth (m)"); ax[1].set_ylabel("particles")
ax[1].legend(frameon=False, fontsize=8)
ax[1].set_title("where the 0.0-floor survivors sit", loc="left", fontsize=10)
for a in ax:
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
    a.grid(axis="y", color="0.92"); a.set_axisbelow(True)
fig.tight_layout(); plt.show()

print(f"\n0.0 floor : {surv0}/{N_PART} survive - frozen at the seafloor W-face")
print(f"NaN floor : {survn}/{N_PART} survive - the rest are DELETED at the bottom")

## What this shows (corrected)

**NaN and 0.0 are identical to Parcels.** Parcels replaces every masked/NaN
value with 0.0 when it loads a field (`field.py:312`). The A/B run confirms it:

    0.0 floor : 100/100 survive (frozen at the seafloor)
    NaN floor : 100/100 survive (frozen at the seafloor)   <- identical

So reverting `Fix_W`'s `fillna(0.0)` to NaN changes nothing - Parcels re-zeros
it. The earlier idea that NaN would delete the particles was wrong. The freezing
is caused by Parcels zeroing the below-seafloor mask (turning land into
0-velocity water) plus the absence of a bottom boundary condition - not by the
`fillna`, and not by the W sign (verified positive-up, correct).

Editing the W field cannot fix this. The options are a boundary treatment in the
run, or - cleaner for publication - truncating each trajectory at seafloor
contact in post-processing (no kernel, pure model advection; see the note added
to the pipeline).
